In [17]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import urllib.request
import zipfile
import os

In [6]:
dataset = load_dataset('stanfordnlp/imdb')
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
train_data = dataset['train']
test_data = dataset['test']

In [12]:
train_data[:10]

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [14]:
from collections import Counter

MAX_VOCAB_SIZE = 20000
MAX_LEN = 200

def tokenize(text):
    return text.lower().split()

counter = Counter()

for example in train_data:
    counter.update(tokenize(example["text"]))

vocab_words = ["<PAD>", "<UNK>"]

vocab_words += [
    word for word, count in counter.most_common(MAX_VOCAB_SIZE - 2)
]

word_to_idx = {
    word: idx for idx, word in enumerate(vocab_words)
}

print("Vocabulary size:", len(word_to_idx))

Vocabulary size: 20000


In [15]:
def encode_text(text):
    tokens = tokenize(text)

    ids = [
        word_to_idx.get(token, word_to_idx["<UNK>"])
        for token in tokens[:MAX_LEN]
    ]

    # Padding
    if len(ids) < MAX_LEN:
        ids += [word_to_idx["<PAD>"]] * (MAX_LEN - len(ids))

    return ids


class IMDBDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        return (
            torch.tensor(encode_text(text), dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )


train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)

In [18]:
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"

if not os.path.exists("glove.6B.100d.txt"):
    urllib.request.urlretrieve(
        glove_url,
        "glove.6B.zip"
    )

    with zipfile.ZipFile("glove.6B.zip", "r") as zip_ref:
        zip_ref.extractall(".")

print("GloVe downloaded!")

GloVe downloaded!


In [19]:
EMBEDDING_DIM = 100

embedding_matrix = np.random.normal(
    0,
    0.6,
    (len(word_to_idx), EMBEDDING_DIM)
)

embedding_matrix[word_to_idx["<PAD>"]] = np.zeros(EMBEDDING_DIM)

with open("glove.6B.100d.txt", "r", encoding="utf-8") as f:

    for line in f:

        values = line.split()

        word = values[0]
        vector = np.asarray(
            values[1:],
            dtype="float32"
        )

        if word in word_to_idx:
            embedding_matrix[word_to_idx[word]] = vector


embedding_matrix = torch.tensor(
    embedding_matrix,
    dtype=torch.float32
)

print(embedding_matrix.shape)

torch.Size([20000, 100])


In [20]:
class SentimentLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers=2,
        pretrained_embeddings=None
    ):

        super().__init__()

        if pretrained_embeddings is not None:

            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings,
                freeze=False,
                padding_idx=word_to_idx["<PAD>"]
            )

        else:

            self.embedding = nn.Embedding(
                vocab_size,
                embedding_dim,
                padding_idx=word_to_idx["<PAD>"]
            )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(
            hidden_dim,
            2
        )

        self.dropout = nn.Dropout(0.3)


    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.dropout(x)

        return self.fc(x)

In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


def train_model(model, epochs=3):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for texts, labels in train_loader:

            texts = texts.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(texts)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Loss: {total_loss / len(train_loader):.4f}"
        )

    return model

Device: cpu


In [22]:
def evaluate_model(model):

    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():

        for texts, labels in test_loader:

            texts = texts.to(device)

            outputs = model(texts)

            preds = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            actual.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        actual,
        predictions
    )

    precision = precision_score(
        actual,
        predictions
    )

    recall = recall_score(
        actual,
        predictions
    )

    f1 = f1_score(
        actual,
        predictions
    )

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    return accuracy, precision, recall, f1

In [23]:
model_random = SentimentLSTM(
    vocab_size=len(word_to_idx),
    embedding_dim=100,
    hidden_dim=128,
    pretrained_embeddings=None
)

print("Training model WITHOUT GloVe...")

model_random = train_model(
    model_random,
    epochs=3
)

print("\nEvaluation:")
random_results = evaluate_model(
    model_random
)

Training model WITHOUT GloVe...


KeyboardInterrupt: 

In [24]:
model_glove = SentimentLSTM(
    vocab_size=len(word_to_idx),
    embedding_dim=100,
    hidden_dim=128,
    pretrained_embeddings=embedding_matrix
)

print("Training model WITH GloVe...")

model_glove = train_model(
    model_glove,
    epochs=3
)

print("\nEvaluation:")
glove_results = evaluate_model(
    model_glove
)

Training model WITH GloVe...


: 

: 

: 